<a href="https://colab.research.google.com/github/akira-kun/SoulX-FlashHead-Colab/blob/main/LatentSync_1.6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 👄 LatentSync 1.6 — Lip Sync no Google Colab

**LatentSync 1.6** (ByteDance) — lip sync via latent diffusion, sem representação intermediária de movimento.

**Destaques da v1.6:**
- ✅ Treinado em 512×512 — resolve o borrado das versões anteriores
- ✅ Requer apenas ~6.5GB de VRAM (cabe no T4 gratuito)
- ✅ Suporta PT-BR, inglês, chinês e outros idiomas
- ✅ Entrada: qualquer vídeo + qualquer áudio

**Nada é salvo no Google Drive.** O vídeo é baixado direto para o seu PC na última célula.

---
⚠️ **Antes de começar:** `Ambiente de execução → Alterar tipo → GPU (T4)`

▶️ **Execute as células em ordem, uma por vez.**

In [1]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 1 — Verificar GPU e ambiente
# ═══════════════════════════════════════════════════════════════
import subprocess, sys, torch

# Verificar GPU
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,compute_cap,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError('Nenhuma GPU detectada! Va em: Ambiente de execucao → Alterar tipo → GPU')

linha = result.stdout.strip().split('\n')[0].split(',')
GPU_NOME = linha[0].strip()
GPU_CAP  = linha[1].strip()
GPU_VRAM = linha[2].strip()
VRAM_GB  = round(int(GPU_VRAM.split()[0]) / 1024, 1)

print('=' * 55)
print(f'  GPU    : {GPU_NOME}')
print(f'  Compute: {GPU_CAP}')
print(f'  VRAM   : {GPU_VRAM} ({VRAM_GB}GB)')
print(f'  Python : {sys.version.split()[0]}')
print(f'  PyTorch: {torch.__version__}')
print(f'  CUDA   : {torch.version.cuda}')
print('=' * 55)

# LatentSync requer ~6.5GB de VRAM
if VRAM_GB < 6:
    print(f'\n⚠️  VRAM insuficiente ({VRAM_GB}GB). LatentSync requer ~6.5GB.')
elif VRAM_GB < 8:
    print(f'\n⚠️  VRAM no limite ({VRAM_GB}GB). Use guidance_scale baixo e inference_steps=20.')
else:
    print(f'\n✅ VRAM suficiente ({VRAM_GB}GB) — LatentSync cabe confortavelmente!')

print('\n✅ Verificacao concluida! Execute a proxima celula.')

  GPU    : Tesla T4
  Compute: 7.5
  VRAM   : 15360 MiB (15.0GB)
  Python : 3.12.13
  PyTorch: 2.10.0+cu128
  CUDA   : 12.8

✅ VRAM suficiente (15.0GB) — LatentSync cabe confortavelmente!

✅ Verificacao concluida! Execute a proxima celula.


In [2]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 2 — Clonar LatentSync e instalar dependências
# Python 3.10 recomendado | CUDA 12.1 recomendado
# ═══════════════════════════════════════════════════════════════
import os
%cd /content

# Clonar repositório
if not os.path.exists('/content/LatentSync'):
    print('Clonando LatentSync...')
    !git clone -q https://github.com/bytedance/LatentSync.git
    print('✅ Repositorio clonado!')
else:
    print('✅ Repositorio ja existe nesta sessao.')

%cd /content/LatentSync

# Instalar ffmpeg
print('\nInstalando ffmpeg...')
!apt-get install -q ffmpeg

# Instalar dependencias do LatentSync
# O requirements.txt inclui: diffusers, transformers, accelerate,
# whisper, decord, imageio, face_alignment, facexlib, etc.
print('\nInstalando dependencias do LatentSync...')
print('Isso pode demorar 3-5 minutos...\n')
!pip install -q -r requirements.txt
!pip install mediapipe==0.10.15

# Garantir versao correta do PyTorch com CUDA
# O Colab atual usa CUDA 12.x — verificar compatibilidade
import torch
if not torch.cuda.is_available():
    print('\n⚠️  CUDA nao disponivel. Reinstalando PyTorch com CUDA...')
    !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
else:
    print(f'\n✅ PyTorch com CUDA ok: {torch.__version__} / CUDA {torch.version.cuda}')

print('\n✅ Dependencias instaladas!')

print("🔧 Instalando dependências faltantes...")

!pip install -q decord omegaconf einops diffusers[torch] transformers accelerate face_alignment
!pip install -q kornia omegaconf einops facexlib face_alignment decord
!pip install -q insightface onnxruntime-gpu
!pip install -q ffmpeg-python librosa soundfile
!pip install -q DeepCache omegaconf einops
!pip install -q jaxlib==0.7.2

print("✅ Bibliotecas instaladas!")


/content
Clonando LatentSync...
✅ Repositorio clonado!
/content/LatentSync

Instalando ffmpeg...
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.

Instalando dependencias do LatentSync...
Isso pode demorar 3-5 minutos...

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 119.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.2 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement mediapipe==0.10.11 (from versions: 0.10.13, 0.10.14, 0.10.15, 0.10.18, 0.10.20, 0.10.21, 0.10.30, 0.10.31, 0.10.32, 0.10.33, 0.10.35)
ERROR: No matching distribution found for mediapipe==0.10.11
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.3 MB/s eta 0:00:00
INFO: pip is looking at multiple vers


✅ PyTorch com CUDA ok: 2.10.0+cu128 / CUDA 12.8

✅ Dependencias instaladas!
🔧 Instalando dependências faltantes...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 86.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.15 requires numpy<2, but you have numpy 2.0.2 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
     ━━━━━━━━━━━━━━━━━━

In [3]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 3 — Baixar checkpoints do LatentSync 1.6
# Ficam em /content/ — NAO ocupam o Google Drive
# Total: ~7GB (unet ~5GB + syncnet ~1.6GB + whisper + auxiliares)
# ═══════════════════════════════════════════════════════════════
import os
from huggingface_hub import snapshot_download
import shutil
import requests
from tqdm import tqdm
import cv2
import numpy as np
from pathlib import Path

%cd /content/LatentSync

CKPT_DIR = '/content/LatentSync/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# Verificar se o arquivo principal já existe para evitar re-download
unet_path = os.path.join(CKPT_DIR, 'latentsync_unet.pt')

if os.path.exists(unet_path):
    print('✅ Checkpoints já existem nesta sessão — pulando download.')
else:
    print('Baixando checkpoints do LatentSync 1.6 (~7GB)...')
    print('Os arquivos ficam em /content/ — não ocupam seu Google Drive.')
    print('Estimativa: 5-10 minutos dependendo da conexão do Colab.\n')

    # Download usando snapshot_download (mais estável que o CLI antigo)
    snapshot_download(
        repo_id="ByteDance/LatentSync-1.6",
        local_dir=CKPT_DIR,
        allow_patterns=[
            "*.pt",
            "*.pth",
            "*.yaml",
            "*.bin",
            "*.json",
            "whisper/**",
            "auxiliary/**"
        ],
        ignore_patterns=["*.md", "*.txt", ".git*"]
    )
    print('\n✅ Checkpoints baixados com sucesso!')

# Criar symlink do VGG16 para o cache do torch (necessário para auxiliares)
vgg_src = os.path.join(CKPT_DIR, 'auxiliary', 'vgg16-397923af.pth')
vgg_dst = os.path.expanduser('~/.cache/torch/hub/checkpoints/vgg16-397923af.pth')
os.makedirs(os.path.dirname(vgg_dst), exist_ok=True)

if os.path.exists(vgg_src) and not os.path.exists(vgg_dst):
    try:
        os.symlink(vgg_src, vgg_dst)
        print('✅ Symlink do VGG16 criado.')
    except Exception as e:
        import shutil
        shutil.copy(vgg_src, vgg_dst)
        print('✅ VGG16 copiado para o cache (fallback).')

# Listar arquivos baixados para conferência
print('\nEstrutura de checkpoints (arquivos principais):')
!find {CKPT_DIR} -maxdepth 2 -name '*.pt' -o -name '*.pth'



# Caminhos
config_origem = '/content/LatentSync/checkpoints/configs/unet/second_stage.yaml'
config_destino = '/content/LatentSync/configs/unet/second_stage.yaml'

# Criar a pasta de destino se não existir
os.makedirs(os.path.dirname(config_destino), exist_ok=True)

# Mover ou copiar o arquivo para onde o código espera
if os.path.exists(config_origem):
    shutil.copy(config_origem, config_destino)
    print("✅ Config UNet movido para o local correto!")
else:
    # Se ele não baixou com o snapshot, vamos baixar manualmente agora
    print("⚠️ Arquivo não encontrado no checkpoint, baixando via wget...")
    !wget -O {config_destino} https://huggingface.co
    print("✅ Config UNet baixado via URL!")

# Verificar se agora está tudo ok
if os.path.exists(config_destino):
    print(f"🚀 Tudo pronto! Configuração em: {config_destino}")

whisper_dir = "/content/LatentSync/checkpoints/whisper"
os.makedirs(whisper_dir, exist_ok=True)
dest_path = os.path.join(whisper_dir, "small.pt")

!wget -O /content/LatentSync/checkpoints/whisper/small.pt https://huggingface.co/jerpint/whisper/resolve/main/small.pt

# Caminho do arquivo que precisa ser alterado
file_to_patch = "/content/LatentSync/latentsync/whisper/whisper/__init__.py"

if os.path.exists(file_to_patch):
    # Lê o conteúdo do arquivo
    with open(file_to_patch, 'r') as f:
        content = f.read()

    # Substitui a trava de segurança por False
    new_content = content.replace("weights_only=True", "weights_only=False")

    # Salva de volta
    with open(file_to_patch, 'w') as f:
        f.write(new_content)

    print("✅ Patch aplicado com sucesso! A trava de segurança do PyTorch foi desativada para o Whisper.")
else:
    print("❌ Arquivo não encontrado. Verifique se o caminho /content/LatentSync existe.")

# 1. Aplicar o Patch de Segurança no UNet (weights_only=False)
unet_script = "/content/LatentSync/latentsync/models/unet.py"
if os.path.exists(unet_script):
    with open(unet_script, 'r') as f:
        content = f.read()
    # Corrige a trava de segurança do PyTorch 2.6 para o UNet
    content = content.replace("weights_only=True", "weights_only=False")
    with open(unet_script, 'w') as f:
        f.write(content)
    print("✅ Patch de segurança aplicado ao UNet.")



# Caminho onde o LatentSync espera as máscaras
mask_dir = "/content/LatentSync/configs/unet"
os.makedirs(mask_dir, exist_ok=True)

# Criar uma máscara branca de 512x512 (indica que a área total pode ser processada)
mask = np.ones((512, 512, 3), dtype=np.uint8) * 255

# Salvar os arquivos que o YAML está pedindo
cv2.imwrite(os.path.join(mask_dir, "mask.png"), mask)
cv2.imwrite(os.path.join(mask_dir, "mask_ratio.png"), mask)

print(f"✅ Máscaras criadas com sucesso em: {mask_dir}")


# Injeta configuração do pipeline para não dar ERROR de OOM CUDA
file_path = Path("/content/LatentSync/scripts/inference.py")
code = file_path.read_text()

# só aplica se ainda NÃO tiver sido modificado
if "enable_attention_slicing" not in code:
    code = code.replace(
        ").to(\"cuda\")",
        ").to(\"cuda\")\n    pipeline.enable_attention_slicing()\n    pipeline.enable_vae_slicing()\n"
    )

    file_path.write_text(code)
    print("✅ Patch aplicado!")
else:
    print("ℹ️ Patch já aplicado, nada a fazer.")




print("📥 Baixando Checkpoint UNet v1.6 (13 canais)...")
# Remove o antigo para não dar conflito
!rm -f /content/LatentSync/checkpoints/latentsync_unet.pt

# Baixa a versão 1.6 real
!wget -O /content/LatentSync/checkpoints/latentsync_unet.pt https://huggingface.co/ByteDance/LatentSync-1.6/resolve/main/latentsync_unet.pt
print("✅ Checkpoint v1.6 baixado!")

/content/LatentSync
Baixando checkpoints do LatentSync 1.6 (~7GB)...
Os arquivos ficam em /content/ — não ocupam seu Google Drive.
Estimativa: 5-10 minutos dependendo da conexão do Colab.



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]


✅ Checkpoints baixados com sucesso!
✅ Symlink do VGG16 criado.

Estrutura de checkpoints (arquivos principais):
/content/LatentSync/checkpoints/stable_syncnet.pt
/content/LatentSync/checkpoints/auxiliary/vgg16-397923af.pth
/content/LatentSync/checkpoints/auxiliary/vit_g_hybrid_pt_1200e_ssv2_ft.pth
/content/LatentSync/checkpoints/auxiliary/i3d_torchscript.pt
/content/LatentSync/checkpoints/auxiliary/sfd_face.pth
/content/LatentSync/checkpoints/latentsync_unet.pt
/content/LatentSync/checkpoints/whisper/tiny.pt
⚠️ Arquivo não encontrado no checkpoint, baixando via wget...
--2026-04-30 17:50:41--  https://huggingface.co/
Resolving huggingface.co (huggingface.co)... 13.249.182.15, 13.249.182.20, 13.249.182.117, ...
Connecting to huggingface.co (huggingface.co)|13.249.182.15|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 147791 (144K) [text/html]
Saving to: ‘/content/LatentSync/configs/unet/second_stage.yaml’

/content/LatentSync 100%[===================>] 144.33K

In [4]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 4 — Upload do VÍDEO
# Suporta 1280x720 e outras resoluções
# ═══════════════════════════════════════════════════════════════
from google.colab import files
import os, subprocess

os.makedirs('/content/inputs', exist_ok=True)
os.makedirs('/content/outputs', exist_ok=True)

print('📁 Faca upload do VÍDEO (MP4, AVI, MOV, MKV)')
print('   Dicas para melhor resultado:')
print('   ✔ Rosto bem visivel e frontal (ou ate 3/4)')
print('   ✔ Boa iluminacao, sem sombras fortes no rosto')
print('   ✔ Resolucoes suportadas: ate 1280x720 e maiores')
print('   ✔ O LatentSync processa a regiao do rosto em 512x512')
print('   ✔ Recomendado: comece com clips de 10-30s para teste\n')

uploaded_video = files.upload()

for fname, content in uploaded_video.items():
    ext = fname.split('.')[-1].lower()
    VIDEO_RAW = f'/content/inputs/video_raw.{ext}'
    with open(VIDEO_RAW, 'wb') as f:
        f.write(content)
    print(f'\n✅ Video salvo: {VIDEO_RAW} ({len(content)/1024/1024:.1f}MB)')
    break

# Inspecionar video e converter para 25fps (requerido pelo LatentSync)
probe = subprocess.run(
    ['ffprobe', '-v', 'quiet', '-show_entries',
     'stream=width,height,r_frame_rate,codec_name:format=duration',
     '-of', 'csv=p=0', VIDEO_RAW],
    capture_output=True, text=True
)

print('\nInfo do video:')
try:
    import cv2
    cap = cv2.VideoCapture(VIDEO_RAW)
    W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    FPS = cap.get(cv2.CAP_PROP_FPS)
    N   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    DUR = N / FPS if FPS > 0 else 0
    cap.release()
    print(f'  Resolucao : {W}x{H}')
    print(f'  FPS       : {FPS:.2f}')
    print(f'  Frames    : {N}')
    print(f'  Duracao   : {DUR:.1f}s')
    if DUR > 60:
        print(f'  ⚠️  Video longo ({DUR:.0f}s). Para teste, considere cortar para 30s.')
except Exception as e:
    print(f'  (Nao foi possivel inspecionar: {e})')

# # LatentSync requer 25fps — converter se necessario
# VIDEO_25FPS = '/content/inputs/video_25fps.mp4'
# print('\nConvertendo video para 25fps (requerido pelo LatentSync)...')
# ret = subprocess.run(
#     ['ffmpeg', '-y', '-i', VIDEO_RAW,
#      '-r', '25',
#      '-c:v', 'libx264', '-crf', '18',
#      '-c:a', 'aac',
#      VIDEO_25FPS],
#     capture_output=True, text=True
# )
# if ret.returncode == 0:
#     print(f'✅ Video convertido para 25fps: {VIDEO_25FPS}')
#     INPUT_VIDEO = VIDEO_25FPS
# else:
#     print(f'⚠️  Conversao falhou — usando video original')
#     print(ret.stderr[-300:])
#     INPUT_VIDEO = VIDEO_RAW

# print(f'\n🎬 Video de entrada: {INPUT_VIDEO}')


# Configurações de otimização para RTX 2060 / FramePack posterior
VIDEO_LOW = '/content/inputs/video_low_res_fps.mp4'
TARGET_FPS = "25"  # FPS reduzido para economizar VRAM
TARGET_HEIGHT = "480" # Resolução 480p

print(f'\n⚙️ Otimizando vídeo: Reduzindo para {TARGET_HEIGHT}p e {TARGET_FPS} fps...')

# Comando ffmpeg para redimensionar (mantendo proporção) e baixar o FPS
ret = subprocess.run(
    ['ffmpeg', '-y', '-i', VIDEO_RAW,
     '-vf', f'scale=-2:{TARGET_HEIGHT}', # -2 mantém a proporção par (exigido pelo h264)
     '-r', TARGET_FPS,
     '-c:v', 'libx264', '-crf', '18',
     '-c:a', 'aac',
     VIDEO_LOW],
    capture_output=True, text=True
)

if ret.returncode == 0:
    print(f'✅ Vídeo otimizado com sucesso: {VIDEO_LOW}')
    INPUT_VIDEO = VIDEO_LOW
else:
    print(f'⚠️ Erro na conversão — Usando vídeo original')
    print(ret.stderr[-300:])
    INPUT_VIDEO = VIDEO_RAW

print(f'\n🎬 Pronto para processar: {INPUT_VIDEO} ({TARGET_HEIGHT}p @ {TARGET_FPS}fps)')

📁 Faca upload do VÍDEO (MP4, AVI, MOV, MKV)
   Dicas para melhor resultado:
   ✔ Rosto bem visivel e frontal (ou ate 3/4)
   ✔ Boa iluminacao, sem sombras fortes no rosto
   ✔ Resolucoes suportadas: ate 1280x720 e maiores
   ✔ O LatentSync processa a regiao do rosto em 512x512
   ✔ Recomendado: comece com clips de 10-30s para teste



Saving 0429.mp4 to 0429.mp4

✅ Video salvo: /content/inputs/video_raw.mp4 (10.5MB)

Info do video:
  Resolucao : 960x720
  FPS       : 30.00
  Frames    : 660
  Duracao   : 22.0s

⚙️ Otimizando vídeo: Reduzindo para 480p e 25 fps...
✅ Vídeo otimizado com sucesso: /content/inputs/video_low_res_fps.mp4

🎬 Pronto para processar: /content/inputs/video_low_res_fps.mp4 (480p @ 25fps)


In [5]:
 ═══════════════════════════════════════════════════════════════
# CÉLULA 5 — Upload do ÁUDIO
# ═══════════════════════════════════════════════════════════════
from google.colab import files
import os, subprocess

print('📁 Faca upload do ÁUDIO (MP3, WAV, AAC, M4A, OGG)')
print('   Dicas:')
print('   ✔ Voz clara, sem muito eco ou ruido de fundo')
print('   ✔ Qualquer idioma funciona — PT-BR incluido!')
print('   ✔ Duracao: idealmente proxima ou maior que a do video')
print('   ✔ Se o audio for mais curto, o video sera cortado automaticamente')
print('   ✔ Formato ideal: WAV 16kHz mono (o notebook converte automaticamente)\n')

uploaded_audio = files.upload()

for fname, content in uploaded_audio.items():
    ext = fname.split('.')[-1].lower()
    AUDIO_RAW = f'/content/inputs/audio_raw.{ext}'
    with open(AUDIO_RAW, 'wb') as f:
        f.write(content)
    print(f'\n✅ Audio salvo: {AUDIO_RAW} ({len(content)/1024:.0f}KB)')
    break

# Converter para WAV 16kHz mono (requerido pelo Whisper do LatentSync)
AUDIO_WAV = '/content/inputs/audio.wav'
print('\nConvertendo audio para WAV 16kHz mono (requerido pelo Whisper)...')
ret = subprocess.run(
    ['ffmpeg', '-y', '-i', AUDIO_RAW,
     '-ar', '16000', '-ac', '1',
     '-acodec', 'pcm_s16le',
     AUDIO_WAV],
    capture_output=True, text=True
)

if ret.returncode == 0:
    # Verificar duracao do audio
    dur_result = subprocess.run(
        ['ffprobe', '-v', 'quiet', '-show_entries', 'format=duration',
         '-of', 'csv=p=0', AUDIO_WAV],
        capture_output=True, text=True
    )
    try:
        dur_audio = float(dur_result.stdout.strip())
        print(f'✅ Audio convertido: {dur_audio:.1f}s a 16kHz mono')
    except:
        print('✅ Audio convertido!')
    INPUT_AUDIO = AUDIO_WAV
else:
    print(f'⚠️  Conversao falhou — usando audio original')
    INPUT_AUDIO = AUDIO_RAW

print(f'\n🎵 Audio de entrada: {INPUT_AUDIO}')

📁 Faca upload do ÁUDIO (MP3, WAV, AAC, M4A, OGG)
   Dicas:
   ✔ Voz clara, sem muito eco ou ruido de fundo
   ✔ Qualquer idioma funciona — PT-BR incluido!
   ✔ Duracao: idealmente proxima ou maior que a do video
   ✔ Se o audio for mais curto, o video sera cortado automaticamente
   ✔ Formato ideal: WAV 16kHz mono (o notebook converte automaticamente)



Saving primeiro-video.mp3 to primeiro-video.mp3

✅ Audio salvo: /content/inputs/audio_raw.mp3 (342KB)

Convertendo audio para WAV 16kHz mono (requerido pelo Whisper)...
✅ Audio convertido: 21.9s a 16kHz mono

🎵 Audio de entrada: /content/inputs/audio.wav


In [6]:
# ═══════════════════════════════════════════════════════════════
# OPCIONAL
# CÉLULA 6 — Configurações de geração manual do yaml
# Ajuste aqui antes de gerar
# ═══════════════════════════════════════════════════════════════
import os

# Caminhos dos modelos
CKPT_DIR      = '/content/LatentSync/checkpoints'
UNET_CONFIG   = '/content/LatentSync/configs/unet/second_stage.yaml'
UNET_CKPT     = f'{CKPT_DIR}/latentsync_unet.pt'
OUTPUT_VIDEO  = '/content/outputs/latentsync_output.mp4'


# !wget -O /content/LatentSync/configs/unet/second_stage.yaml https://raw.githubusercontent.com/bytedance/LatentSync/refs/heads/main/configs/unet/stage2_512.yaml

# # Conteúdo correto do YAML para LatentSync 1.6 (usando apenas espaços)
# os.makedirs(os.path.dirname(UNET_CONFIG), exist_ok=True)
# # Conteúdo formatado rigorosamente com espaços duplos, sem nenhum TAB
# # Conteúdo puro sem nenhuma indentação automática de editores
# # O segredo aqui é colocar os colchetes [25] no audio_feat_length
# yaml_content = """data:
#   syncnet_config_path: configs/syncnet/syncnet_16_pixel_attn.yaml
#   train_output_dir: debug/unet
#   train_fileslist: /mnt/bn/maliva-gen-ai-v2/chunyu.li/fileslist/data_v10_core.txt
#   train_data_dir: ""
#   audio_embeds_cache_dir: /mnt/bn/maliva-gen-ai-v2/chunyu.li/audio_cache/embeds
#   audio_mel_cache_dir: /mnt/bn/maliva-gen-ai-v2/chunyu.li/audio_cache/mel

#   val_video_path: assets/demo1_video.mp4
#   val_audio_path: assets/demo1_audio.wav
#   batch_size: 1 # 4
#   num_workers: 12 # 12
#   num_frames: 16
#   resolution: 256
#   mask_image_path: latentsync/utils/mask.png
#   audio_sample_rate: 16000
#   video_fps: 25
#   audio_feat_length: [2, 2]

# ckpt:
#   resume_ckpt_path: checkpoints/latentsync_unet.pt
#   save_ckpt_steps: 10000

# run:
#   pixel_space_supervise: true
#   use_syncnet: true
#   sync_loss_weight: 0.05
#   perceptual_loss_weight: 0.1 # 0.1
#   recon_loss_weight: 1 # 1
#   guidance_scale: 1.5 # [1.0 - 3.0]
#   trepa_loss_weight: 0
#   inference_steps: 20
#   trainable_modules:
#     - motion_modules.
#     - attn2.
#   seed: 1247
#   use_mixed_noise: true
#   mixed_noise_alpha: 1 # 1
#   mixed_precision_training: true
#   enable_gradient_checkpointing: true
#   max_train_steps: 10000000
#   max_train_epochs: -1

# optimizer:
#   lr: 1e-5
#   scale_lr: false
#   max_grad_norm: 1.0
#   lr_scheduler: constant
#   lr_warmup_steps: 0

# model:
#   act_fn: silu
#   add_audio_layer: true
#   attention_head_dim: 8
#   block_out_channels: [320, 640, 1280, 1280]
#   center_input_sample: false
#   cross_attention_dim: 384
#   down_block_types:
#     [
#       "CrossAttnDownBlock3D",
#       "CrossAttnDownBlock3D",
#       "CrossAttnDownBlock3D",
#       "DownBlock3D",
#     ]
#   mid_block_type: UNetMidBlock3DCrossAttn
#   up_block_types:
#     [
#       "UpBlock3D",
#       "CrossAttnUpBlock3D",
#       "CrossAttnUpBlock3D",
#       "CrossAttnUpBlock3D",
#     ]
#   downsample_padding: 1
#   flip_sin_to_cos: true
#   freq_shift: 0
#   in_channels: 13 # 49
#   layers_per_block: 2
#   mid_block_scale_factor: 1
#   norm_eps: 1e-5
#   norm_num_groups: 32
#   out_channels: 4 # 16
#   sample_size: 64
#   resnet_time_scale_shift: default # Choose between [default, scale_shift]

#   use_motion_module: true
#   motion_module_resolutions: [1, 2, 4, 8]
#   motion_module_mid_block: false
#   motion_module_decoder_only: true
#   motion_module_type: Vanilla
#   motion_module_kwargs:
#     num_attention_heads: 8
#     num_transformer_block: 1
#     attention_block_types:
#       - Temporal_Self
#       - Temporal_Self
#     temporal_position_encoding: true
#     temporal_position_encoding_max_len: 24
#     temporal_attention_dim_div: 1
#     zero_initialize: true
# """

# with open(UNET_CONFIG, "w", encoding="utf-8") as f:
#     f.write(yaml_content.strip())


# print(f"✅ Arquivo {UNET_CONFIG} corrigido e salvo sem TABs!")



# Verificar se os arquivos existem
checks = {
    'UNet checkpoint': UNET_CKPT,
    'Whisper tiny':    f'{CKPT_DIR}/whisper/tiny.pt',
    'Video entrada':   INPUT_VIDEO,
    'Audio entrada':   INPUT_AUDIO,
    'Config UNet':     UNET_CONFIG,
}

print('Verificando arquivos necessarios:')
todos_ok = True
for nome, caminho in checks.items():
    existe = os.path.exists(caminho)
    status = '✅' if existe else '❌'
    tamanho = f'({os.path.getsize(caminho)/1024/1024:.1f}MB)' if existe else ''
    print(f'  {status} {nome}: {caminho} {tamanho}')
    if not existe:
        todos_ok = False

print()
if not todos_ok:
    print('❌ Alguns arquivos estao faltando!')
    print('   - Se o checkpoint falta: rode a celula 3 novamente')
    print('   - Se video/audio falta: rode as celulas 4/5 novamente')
else:
    print('✅ Tudo pronto! Configuracoes:')
    print(f'  Steps         : {INFERENCE_STEPS}')
    print(f'  Guidance scale: {GUIDANCE_SCALE}')
    print(f'  Seed          : {SEED}')
    print(f'  Bbox shift    : {BBOX_SHIFT}')
    print(f'  Saida         : {OUTPUT_VIDEO}')
    print(f'  Drive usado   : NAO — tudo em /content/')
    print('\n  Execute a celula 8 para gerar!')

mask_image_path: /content/LatentSync/configs/unet/mask.png

✅ Verificando todas as chaves do inference.py:
  config.model.cross_attention_dim : 768
  config.data.num_frames           : 16
  config.data.audio_feat_length    : [2, 2]
  config.data.resolution           : 256
  config.data.mask_image_path      : /content/LatentSync/configs/unet/mask.png
  down_block_types                 : ['CrossAttnDownBlock3D', 'CrossAttnDownBlock3D', 'CrossAttnDownBlock3D', 'DownBlock3D']
  up_block_types                   : ['UpBlock3D', 'CrossAttnUpBlock3D', 'CrossAttnUpBlock3D', 'CrossAttnUpBlock3D']
  mask existe em disco             : True

🚀 Execute a célula de inferência agora!
Verificando arquivos necessarios:
  ✅ UNet checkpoint: /content/LatentSync/checkpoints/latentsync_unet.pt (4837.2MB)
  ✅ Whisper tiny: /content/LatentSync/checkpoints/whisper/tiny.pt (72.1MB)
  ✅ Video entrada: /content/inputs/video_low_res_fps.mp4 (1.9MB)
  ✅ Audio entrada: /content/inputs/audio.wav (0.7MB)
  ✅ Config UN

In [21]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 6 CORRIGIDA — Baixar e Adaptar o YAML (stage2_512.yaml)
# ═══════════════════════════════════════════════════════════════
import os
import yaml
import subprocess
import cv2
import numpy as np

# Caminhos
CKPT_DIR = '/content/LatentSync/checkpoints'
UNET_CONFIG = '/content/LatentSync/configs/unet/second_stage.yaml' # Nome que o inference.py espera
UNET_CKPT = f'{CKPT_DIR}/latentsync_unet.pt'
OUTPUT_VIDEO = '/content/outputs/latentsync_output.mp4'

# --- 1. Baixar o arquivo de configuração CORRETO (stage2.yaml) ---
print("📥 Baixando arquivo de configuração CORRETO (stage2.yaml)...")
os.makedirs(os.path.dirname(UNET_CONFIG), exist_ok=True)
# URL do arquivo que você encontrou
yaml_url = "https://raw.githubusercontent.com/bytedance/LatentSync/refs/heads/main/configs/unet/stage2_512.yaml"
subprocess.run(['wget', '-O', UNET_CONFIG, yaml_url], check=True)
print("✅ Arquivo stage2.yaml baixado com sucesso!")

# --- 2. Adaptar o YAML para Inferência ---
print("🔧 Adaptando o arquivo YAML para inferência...")
with open(UNET_CONFIG, 'r') as f:
    config = yaml.safe_load(f)

# Ajustes necessários:
# 1. O script inference.py espera a chave 'mask_image_path' no nível superior.
#    Vamos extrair do 'data' se existir, ou definir um padrão.
mask_path_in_config = "/content/LatentSync/latentsync/utils/mask.png"
if 'data' in config and 'mask_image_path' in config['data']:
    # Se o caminho relativo existir, converte para absoluto
    rel_path = config['data']['mask_image_path']
    if not os.path.isabs(rel_path):
        mask_path_in_config = os.path.join('/content/LatentSync', rel_path)
    else:
        mask_path_in_config = rel_path
    print(f"   Caminho da máscara encontrado no YAML: {mask_path_in_config}")

# Garantir que a máscara exista
if not os.path.exists(mask_path_in_config):
    print(f"⚠️ Máscara não encontrada em {mask_path_in_config}. Criando uma máscara padrão (branca)...")
    os.makedirs(os.path.dirname(mask_path_in_config), exist_ok=True)
    mask = np.ones((256, 256, 3), dtype=np.uint8) * 255
    cv2.imwrite(mask_path_in_config, mask)
    print(f"✅ Máscara criada em {mask_path_in_config}")

# O script inference.py também espera outras chaves no nível superior.
# Vamos criar um dicionário de configuração para inferência.
inference_config = {
    'model': config.get('model', {}),
    'noise_scheduler': config.get('noise_scheduler', {}),
    'data': config.get('data', {}) # O inference.py pode usar algumas chaves daqui também
}

# Salvar o arquivo adaptado
with open(UNET_CONFIG, 'w') as f:
    yaml.dump(inference_config, f, default_flow_style=False)

print(f"✅ Arquivo de configuração adaptado salvo em: {UNET_CONFIG}")

# Verificação rápida
print("\n✅ Verificando se as chaves principais existem:")
with open(UNET_CONFIG, 'r') as f:
    new_config = yaml.safe_load(f)
    print(f"   - 'model' presente: {'model' in new_config}")
    print(f"   - 'noise_scheduler' presente: {'noise_scheduler' in new_config}")
    print(f"   - 'data' presente: {'data' in new_config}")

print("\n🚀 Configuração pronta! Execute a CÉLULA 7 (Inferência Corrigida) agora.")

📥 Baixando arquivo de configuração CORRETO (stage2.yaml)...
✅ Arquivo stage2.yaml baixado com sucesso!
🔧 Adaptando o arquivo YAML para inferência...
   Caminho da máscara encontrado no YAML: /content/LatentSync/latentsync/utils/mask.png
✅ Arquivo de configuração adaptado salvo em: /content/LatentSync/configs/unet/second_stage.yaml

✅ Verificando se as chaves principais existem:
   - 'model' presente: True
   - 'noise_scheduler' presente: True
   - 'data' presente: True

🚀 Configuração pronta! Execute a CÉLULA 7 (Inferência Corrigida) agora.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 7 CORRIGIDA — com resolution correta
# ═══════════════════════════════════════════════════════════════
import os, time, subprocess, torch, gc

%cd /content/LatentSync
os.environ['PYTHONPATH'] = '/content/LatentSync'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
torch.cuda.empty_cache()

# Caminhos
UNET_CONFIG  = '/content/LatentSync/configs/unet/second_stage.yaml'
UNET_CKPT    = '/content/LatentSync/checkpoints/latentsync_unet.pt'
INPUT_VIDEO  = '/content/inputs/video_low_res_fps.mp4'
INPUT_AUDIO  = '/content/inputs/audio.wav'
OUTPUT_VIDEO = '/content/outputs/latentsync_output.mp4'

# Parâmetros — ajuste aqui
INFERENCE_STEPS = 20
GUIDANCE_SCALE  = 2.0   # aumentado para forçar mais o lipsync
SEED            = 1234
BBOX_SHIFT      = -7    # negativo abre mais a região da boca

# Verificar resolução real do vídeo e ajustar o yaml
import cv2, yaml
cap = cv2.VideoCapture(INPUT_VIDEO)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f'Resolução do vídeo: {W}x{H}')

# O resolution no yaml deve ser 512 (tamanho interno do crop do rosto)
# mas precisamos garantir que o whisper model correto está sendo usado
# Verificar qual whisper está disponível
import os
ckpt_dir = '/content/LatentSync/checkpoints/whisper'
whisper_files = os.listdir(ckpt_dir) if os.path.exists(ckpt_dir) else []
print(f'Whisper disponível: {whisper_files}')

# Se tiver small.pt (461MB) foi baixado o modelo errado — deve ser tiny.pt
# O inference.py carrega pelo nome hardcoded 'tiny.pt' ou pelo arg --whisper_model
whisper_model = 'small' if 'small.pt' in whisper_files else 'tiny'
print(f'Usando whisper model: {whisper_model}')

# Montar comando COMPLETO com todos os parâmetros
cmd = (
    f'python -m scripts.inference '
    f'--unet_config_path {UNET_CONFIG} '
    f'--inference_ckpt_path {UNET_CKPT} '
    f'--video_path {INPUT_VIDEO} '
    f'--audio_path {INPUT_AUDIO} '
    f'--video_out_path {OUTPUT_VIDEO} '
    f'--guidance_scale {GUIDANCE_SCALE} '
    f'--inference_steps {INFERENCE_STEPS} '
    f'--seed {SEED} '
)

print(f'\n🚀 Executando inferência...')
print(f'   Steps: {INFERENCE_STEPS} | Guidance: {GUIDANCE_SCALE} | BBox: {BBOX_SHIFT}')
print(f'   Vídeo: {W}x{H}px')
print(f'\nComando:\n{cmd}\n')
print('-' * 55)

inicio = time.time()
resultado = subprocess.run(cmd, shell=True, capture_output=True, text=True)
elapsed = time.time() - inicio

print('-' * 55)
print(f'Tempo: {elapsed:.1f}s ({elapsed/60:.1f} min)')

if resultado.returncode != 0:
    print('\n❌ FALHOU! Erro:')
    print(resultado.stderr[-3000:])
else:
    # Mostrar stdout para ver progresso
    print(resultado.stdout[-2000:])
    print('✅ Concluído!')

# Verificar resultado
if os.path.exists(OUTPUT_VIDEO) and os.path.getsize(OUTPUT_VIDEO) > 0:
    mb = os.path.getsize(OUTPUT_VIDEO) / (1024*1024)
    print(f'\n✅ Vídeo gerado: {OUTPUT_VIDEO} ({mb:.1f}MB)')
    GERADO_OK = True
else:
    GERADO_OK = False
    print('\n⚠️  Vídeo não encontrado.')
    # Verificar se o stdout tem pistas
    print('\nSTDOUT (primeiras linhas):')
    print(resultado.stdout[:1000])

/content/LatentSync
Resolução do vídeo: 640x480
Whisper disponível: ['tiny.pt', 'small.pt']
Usando whisper model: small

🚀 Executando inferência...
   Steps: 20 | Guidance: 2.0 | BBox: -7
   Vídeo: 640x480px

Comando:
python -m scripts.inference --unet_config_path /content/LatentSync/configs/unet/second_stage.yaml --inference_ckpt_path /content/LatentSync/checkpoints/latentsync_unet.pt --video_path /content/inputs/video_low_res_fps.mp4 --audio_path /content/inputs/audio.wav --video_out_path /content/outputs/latentsync_output.mp4 --guidance_scale 2.0 --inference_steps 20 --seed 1234 

-------------------------------------------------------


In [20]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 8 — Visualizar resultado + DOWNLOAD AUTOMÁTICO
# NAO salva no Google Drive — baixa direto para o seu PC
# ═══════════════════════════════════════════════════════════════
from google.colab import files
from IPython.display import HTML, display
from base64 import b64encode
import os

# Procurar o video gerado
OUTPUT_DIR = '/content/outputs'
videos = sorted(
    [os.path.join(OUTPUT_DIR, f) for f in os.listdir(OUTPUT_DIR)
     if f.endswith('.mp4') and os.path.getsize(os.path.join(OUTPUT_DIR, f)) > 0],
    key=os.path.getmtime, reverse=True
)

if not videos:
    print('⚠️  Nenhum video para exibir. Execute a celula 7 primeiro.')
else:
    video_path = videos[0]
    nome = os.path.basename(video_path)
    mb   = os.path.getsize(video_path) / (1024*1024)

    print(f'Reproduzindo: {nome} ({mb:.1f}MB)')
    print('-' * 50)

    # Exibir player inline
    with open(video_path, 'rb') as f:
        b64 = b64encode(f.read()).decode()

    display(HTML(f'''
    <video width="720" controls style="border-radius:8px;margin:12px 0;max-width:100%">
        <source src="data:video/mp4;base64,{b64}" type="video/mp4">
    </video>
    <p style="font-family:monospace;color:#555;font-size:13px">
        📹 {nome} &nbsp;|&nbsp; {mb:.1f}MB &nbsp;|&nbsp;
        LatentSync 1.6 &nbsp;|&nbsp;
        Steps: {INFERENCE_STEPS} &nbsp;|&nbsp;
        Guidance: {GUIDANCE_SCALE}
    </p>
    '''))

    # Download automatico
    print('\n⬇️  Iniciando download automatico para o seu PC...')
    files.download(video_path)
    print('✅ Download iniciado! Verifique a pasta de Downloads.')
    print('\n💡 Nada foi salvo no Google Drive — so no seu PC.')
    print('\n─────────────────────────────────────────────────')
    print('Para gerar outro video, execute apenas as celulas 4, 5 e 7.')
    print('Os modelos ja estao carregados — nao precisa re-baixar.')

Reproduzindo: latentsync_output.mp4 (1.8MB)
--------------------------------------------------



⬇️  Iniciando download automatico para o seu PC...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download iniciado! Verifique a pasta de Downloads.

💡 Nada foi salvo no Google Drive — so no seu PC.

─────────────────────────────────────────────────
Para gerar outro video, execute apenas as celulas 4, 5 e 7.
Os modelos ja estao carregados — nao precisa re-baixar.


---
## 🔄 Gerar novo vídeo na mesma sessão

Execute apenas as células **4, 5, 7 e 8** com novos arquivos.
Modelos já estão carregados — não precisa re-baixar.

---
## ⚙️ Parâmetros principais (célula 6)

| Parâmetro | Valor | Efeito |
|---|---|---|
| `INFERENCE_STEPS` | 20 | Balanceado (rápido) |
| `INFERENCE_STEPS` | 30–50 | Mais qualidade, mais lento |
| `GUIDANCE_SCALE` | 1.0 | Movimento mais natural |
| `GUIDANCE_SCALE` | 1.5 | Lip sync mais preciso |
| `BBOX_SHIFT` | 0 | Padrão |
| `BBOX_SHIFT` | -7 | Abre mais a boca (recomendado para PT-BR) |
| `BBOX_SHIFT` | +5 | Fecha mais a boca |

---
## 📋 Solução de problemas

| Problema | Solução |
|---|---|
| `CUDA out of memory` | Reduza `INFERENCE_STEPS` para 10. Reinicie o runtime |
| `No face detected` | Rosto deve estar bem visivel e frontal. Tente `BBOX_SHIFT = -10` |
| Lip sync impreciso | Aumente `GUIDANCE_SCALE` para 1.5–2.0. Diminua `BBOX_SHIFT` |
| Boca muito fechada | Use `BBOX_SHIFT = -5` ou `-10` |
| Sessão expirou | Execute do início — células 1-3 levam ~10 min para reinstalar |
| Erro de imports jax | Ignorar — nao afeta o LatentSync (conflito com o Colab nativo) |

---
## ⚡ Tempos estimados no Colab

| GPU | 10s de vídeo | 30s de vídeo |
|---|---|---|
| **T4** (gratuito) | ~3–5 min | ~8–15 min |
| **L4** (Pro) | ~1–2 min | ~4–6 min |
| **A100** (Pro+) | ~30–60s | ~2–3 min |

---
## 🔗 Links
- GitHub: https://github.com/bytedance/LatentSync
- HuggingFace: https://huggingface.co/ByteDance/LatentSync-1.6
- Paper: https://arxiv.org/abs/2412.09262